In [2]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime
from typing import Any

import anndata as ad

LOG_KEY = "_omicslog"


def _timestamp() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def _format_log_message(operation: str, message: str, ts: str | None = None) -> str:
    stamp = ts or _timestamp()
    return f"[{stamp}] {operation}: {message}"


def _ensure_log_container(adata: ad.AnnData) -> list[str]:
    current = adata.uns.get(LOG_KEY)
    if not isinstance(current, list):
        adata.uns[LOG_KEY] = []
    return adata.uns[LOG_KEY]


def _append_log_messages(adata: ad.AnnData, messages: list[str] | tuple[str, ...]) -> None:
    if not messages:
        return
    _ensure_log_container(adata).extend(messages)


def _inherit_and_append(
    parent: ad.AnnData,
    child: ad.AnnData,
    messages: list[str] | tuple[str, ...],
) -> None:
    child.uns[LOG_KEY] = list(_ensure_log_container(parent))
    _append_log_messages(child, messages)


def _subset_messages(
    pre_shape: tuple[int, int],
    post_shape: tuple[int, int],
    operation: str = "subset",
) -> list[str]:
    msgs: list[str] = []
    ts = _timestamp()

    pre_obs, pre_var = pre_shape
    post_obs, post_var = post_shape

    if pre_var != post_var:
        removed_genes = pre_var - post_var
        percent_removed = round((removed_genes / pre_var) * 100) if pre_var else 0
        msgs.append(
            _format_log_message(
                operation,
                f"removed {removed_genes} genes ({percent_removed}%), {post_var} genes remaining",
                ts,
            )
        )

    if pre_obs != post_obs:
        removed_samples = pre_obs - post_obs
        percent_removed = round((removed_samples / pre_obs) * 100) if pre_obs else 0
        msgs.append(
            _format_log_message(
                operation,
                (
                    f"removed {removed_samples} samples ({percent_removed}%), "
                    f"{post_obs} samples remaining"
                ),
                ts,
            )
        )
    return msgs


class LoggedAnnDataStandalone(ad.AnnData):
    """Standalone subclass strategy with local logging helpers and message style."""

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        _ensure_log_container(self)

    @classmethod
    def _safe_component_copy(cls, value):
        return value.copy() if hasattr(value, "copy") else deepcopy(value)

    @classmethod
    def from_anndata(cls, adata: ad.AnnData) -> "LoggedAnnDataStandalone":
        if isinstance(adata, cls):
            _ensure_log_container(adata)
            return adata

        kwargs: dict[str, Any] = {
            "X": cls._safe_component_copy(adata.X) if adata.X is not None else None,
            "obs": adata.obs.copy(),
            "var": adata.var.copy(),
            "uns": deepcopy(dict(adata.uns)),
            "obsm": {k: cls._safe_component_copy(v) for k, v in adata.obsm.items()},
            "varm": {k: cls._safe_component_copy(v) for k, v in adata.varm.items()},
            "layers": {k: cls._safe_component_copy(v) for k, v in adata.layers.items()},
            "obsp": {k: cls._safe_component_copy(v) for k, v in adata.obsp.items()},
            "varp": {k: cls._safe_component_copy(v) for k, v in adata.varp.items()},
        }
        if adata.raw is not None:
            kwargs["raw"] = {
                "X": cls._safe_component_copy(adata.raw.X),
                "var": adata.raw.var.copy(),
                "varm": {k: cls._safe_component_copy(v) for k, v in adata.raw.varm.items()},
            }

        logged = cls(**kwargs)
        _ensure_log_container(logged)
        return logged

    def __getitem__(self, index):
        pre_shape = self.shape
        result = super().__getitem__(index)
        logged_result = self.from_anndata(result)
        msgs = _subset_messages(pre_shape, logged_result.shape, operation="subset")
        _inherit_and_append(self, logged_result, msgs)
        return logged_result

    def _inplace_subset_obs(self, index):
        pre_shape = self.shape
        super()._inplace_subset_obs(index)
        _append_log_messages(self, _subset_messages(pre_shape, self.shape, operation="subset"))

    def _inplace_subset_var(self, index):
        pre_shape = self.shape
        super()._inplace_subset_var(index)
        _append_log_messages(self, _subset_messages(pre_shape, self.shape, operation="subset"))

    def _operation_log_block(self) -> str:
        logs = self.uns.get(LOG_KEY, [])
        if not logs:
            return ""
        return "\n\nOperation log:\n" + "\n".join(str(x) for x in logs)

    def __repr__(self) -> str:
        return super().__repr__() + self._operation_log_block()

    def __str__(self) -> str:
        return self.__repr__()

    def operation_log(self) -> list[str]:
        return list(self.uns.get(LOG_KEY, []))


def log_start(adata: ad.AnnData) -> LoggedAnnDataStandalone:
    return LoggedAnnDataStandalone.from_anndata(adata)


In [3]:
logged1 = log_start(adata1)

# User-style filtering
logged1[logged1.obs["dex"] == "untrt", :]



NameError: name 'adata1' is not defined